# 🧹 Project: Data Cleanser

## 📌 Part A: Handling Missing Values

### 📚 Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer

### 📂 Load the Dataset

In [2]:
df = pd.read_csv("../Dataset/data_cleanser_dataset.csv")
df

,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1,63.0,NaN,West,49.9,151.7,NaN,NaN,1
1,2,46.0,Male,East,34.0,150.3,232.1,74.1,1
2,3,42.0,Male,North,22.5,124.4,148.9,359.5,1
3,4,34.0,Female,North,29.6,159.0,305.5,129.0,1
4,5,NaN,Female,North,28.4,95.0,210.8,NaN,0
...,...,...,...,...,...,...,...,...,...
1995,1996,NaN,Female,East,40.3,155.2,156.9,92.7,1
1996,1997,78.0,Female,North,26.8,158.5,249.9,79.8,1
1997,1998,74.0,NaN,East,34.0,153.4,240.5,128.8,1
1998,1999,72.0,NaN,North,22.1,111.1,223.9,84.2,0


### 🎯 Shape

In [3]:
df.shape

(2000, 9)

### 📋 Data Types

In [4]:
df.dtypes

patient_id          int64
age               float64
gender             object
region             object
bmi               float64
blood_pressure    float64
cholesterol       float64
glucose           float64
disease_risk        int64
dtype: object

### 🔍 Check Missing Values

In [5]:
missing_values = df.isnull().sum()
missing_values

patient_id          0
age               196
gender            658
region            386
bmi               160
blood_pressure      0
cholesterol       171
glucose           164
disease_risk        0
dtype: int64

### 📊 Calculate Missing Percentage

In [ ]:
(df.isnull().sum() / len(df)) * 100missing_percentage = 
missing_percentage

patient_id         0.00
age                9.80
gender            32.90
region            19.30
bmi                8.00
blood_pressure     0.00
cholesterol        8.55
glucose            8.20
disease_risk       0.00
dtype: float64

### 📋 Create Summary Report

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Values":missing_values,
    "Missing Percentage": missing_percentage
})
missing_summary

,Missing Values,Missing Percentage
patient_id,0,0.00
age,196,9.80
gender,658,32.90
region,386,19.30
bmi,160,8.00
blood_pressure,0,0.00
cholesterol,171,8.55
glucose,164,8.20
disease_risk,0,0.00


### 🧮 Simple Imputer (Numerical)

In [8]:
df_mean = df.copy()

mean_imputer = SimpleImputer(strategy="mean")
df_mean["bmi"] = mean_imputer.fit_transform(df_mean[["bmi"]]).ravel()

In [9]:
df_mean["bmi"].isnull().sum()

np.int64(0)

### 🏷️ Simple Imputer (Categorical)

In [10]:
df_region = df.copy()

region_imputer = SimpleImputer(strategy="most_frequent")
df_region["region"] = region_imputer.fit_transform(df_region[["region"]]).ravel()

In [11]:
df_region["region"].isnull().sum()

np.int64(0)

### 👤 Most Frequent Imputation

In [12]:
df_gender = df.copy()

gender_imputer = SimpleImputer(strategy="most_frequent")
df_gender["gender"] = gender_imputer.fit_transform(df_gender[["gender"]]).ravel()

In [13]:
df_gender["gender"].isnull().sum()

np.int64(0)

### 🎯 Missing Indicator + Random Sample Imputation

In [14]:
df_random = df.copy()

# Create missing indicator
df_random["cholesterol_missing"] = df_random["cholesterol"].isnull().astype(int)

In [15]:
# Random Sample Imputation
random_values = df_random["cholesterol"].dropna().sample(
    df_random["cholesterol"].isnull().sum(),
    random_state=42,
    replace=True
).values

In [ ]:
df_random.loc[df_random["cholesterol"].isnull(), "cholesterol"] = random_values

In [17]:
df_random["cholesterol"].isnull().sum()

np.int64(0)

### 🤝 KNN Imputer

In [18]:
df_knn = df.copy()

# Select numerical columns
num_cols = ["age", "bmi", "blood_pressure", "cholesterol", "glucose"]

In [19]:
# Create KNN Imputer
knn = KNNImputer(n_neighbors=5)

In [20]:
# Apply KNN
df_knn[num_cols] = knn.fit_transform(df_knn[num_cols])

In [21]:
df_knn[num_cols].isnull().sum()

age               0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
dtype: int64

### 🧠 MICE Algorithm

In [22]:
df_mice = df.copy()

# Select numerical columns
num_cols = ["age", "bmi", "blood_pressure", "cholesterol", "glucose"]

In [23]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
mice = IterativeImputer(random_state=42)

In [24]:
# Apply MICE
df_mice[num_cols] = mice.fit_transform(df_mice[num_cols])

In [25]:
df_mice[num_cols].isnull().sum()

age               0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
dtype: int64

## 📌 Part B: Handling Outliers

### 📈 Z-Score Method

In [26]:
from scipy.stats import zscore

clean_data = df_mice.copy()

z_cols = ["cholesterol", "glucose"]      # Select numerical columns

z_scores = np.abs(zscore(clean_data[z_cols]))     # Calculate Z-score

data_z = clean_data[(z_scores < 3).all(axis=1)]     # Remove outliers

print("Before Z-score:", clean_data.shape)
print("After Z-score:", data_z.shape)

Before Z-score: (2000, 9)
After Z-score: (1870, 9)


### 📦 IQR Method

In [27]:
data_iqr = clean_data.copy()

Q1 = data_iqr["bmi"].quantile(0.25)     # Calculate Q1 and Q3
Q3 = data_iqr["bmi"].quantile(0.75)

IQR = Q3 - Q1    # Calculate IQR

lower_limit = Q1 - 1.5 * IQR     # Calculate lower and upper limits
upper_limit = Q3 + 1.5 * IQR

bmi_outliers = data_iqr[      # Detect BMI outliers
    (data_iqr["bmi"] < lower_limit) |
    (data_iqr["bmi"] > upper_limit)
]

print("Number of BMI Outliers:", len(bmi_outliers))

Number of BMI Outliers: 22


### 📉 Percentile Method

In [28]:
data_pct = clean_data.copy()

lower = data_pct[num_cols].quantile(0.01)    # Calculate lower and upper percentile values
upper = data_pct[num_cols].quantile(0.99)

data_pct[num_cols] = data_pct[num_cols].clip(lower, upper, axis=1)   # Apply percentile capping

data_pct.describe()

,patient_id,age,bmi,blood_pressure,cholesterol,glucose,disease_risk
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,1000.500000,51.848425,30.425059,130.149270,205.569429,115.107979,0.751500
std,577.494589,18.756144,8.605422,28.211103,45.732342,46.975217,0.432251
min,1.000000,18.000000,16.300000,90.700000,141.400000,70.999000,0.000000
25%,500.750000,37.000000,23.500000,108.675000,173.000000,90.075000,1.000000
50%,1000.500000,51.843341,30.408701,127.750000,204.617427,110.200000,1.000000
75%,1500.250000,67.000000,36.400000,146.125000,228.100000,123.225000,1.000000
max,2000.000000,85.000000,56.300000,233.402000,388.300000,353.014000,1.000000


### ✂️ Winsorization

In [29]:
from scipy.stats.mstats import winsorize

data_win = clean_data.copy()

# Apply Winsorization
for col in num_cols:
    data_win[col] = winsorize(data_win[col], limits=[0.05, 0.05])

data_win.describe()

c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the 

,patient_id,age,bmi,blood_pressure,cholesterol,glucose,disease_risk
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,1000.500000,51.838425,29.980859,129.055150,201.419229,107.389849,0.751500
std,577.494589,18.477354,7.500297,24.577345,33.767952,20.178108,0.432251
min,1.000000,21.000000,17.500000,93.900000,145.700000,74.000000,0.000000
25%,500.750000,37.000000,23.500000,108.675000,173.000000,90.075000,1.000000
50%,1000.500000,51.843341,30.408701,127.750000,204.617427,110.200000,1.000000
75%,1500.250000,67.000000,36.400000,146.125000,228.100000,123.225000,1.000000
max,2000.000000,82.000000,41.900000,190.700000,258.600000,139.800000,1.000000


### 📊 Compare Dataset Before vs After Outlier Treatment

In [30]:
print("Before Outlier Treatment Shape:", clean_data.shape)
print("After Outlier Treatment Shape:", data_win.shape)

Before Outlier Treatment Shape: (2000, 9)
After Outlier Treatment Shape: (2000, 9)


In [31]:
print("Before Outlier Treatment")
print(clean_data.describe())

print("After Outlier Treatment")
print(data_win.describe())

Before Outlier Treatment
        patient_id          age          bmi  blood_pressure  cholesterol  \
count  2000.000000  2000.000000  2000.000000     2000.000000  2000.000000   
mean   1000.500000    51.848425    30.440059      130.178350   205.702829   
std     577.494589    18.756144     8.660933       28.333406    46.332951   
min       1.000000    18.000000    16.000000       90.000000   140.000000   
25%     500.750000    37.000000    23.500000      108.675000   173.000000   
50%    1000.500000    51.843341    30.408701      127.750000   204.617427   
75%    1500.250000    67.000000    36.400000      146.125000   228.100000   
max    2000.000000    85.000000    59.900000      239.700000   415.100000   

           glucose  disease_risk  
count  2000.000000   2000.000000  
mean    115.246099      0.751500  
std      47.734422      0.432251  
min      70.000000      0.000000  
25%      90.075000      1.000000  
50%     110.200000      1.000000  
75%     123.225000      1.000000  
m

c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the 

## 📌 Part C: Final Clean Dataset

In [32]:
final_data = data_win.copy()

final_data.to_csv("../Dataset/Final_Clean_Dataset.csv", index=False)
print("Final cleaned dataset saved successfully.")

Final cleaned dataset saved successfully.


### 📝 Brief Report

#### ❓ 1. Which imputation strategy was most effective?

**Answer:**

MICE (Multiple Imputation by Chained Equations) was the most effective imputation strategy. It estimates missing values using the relationships between different variables, which makes the imputed values more accurate than simple methods like mean or most frequent imputation.

---

#### ❓ 2. Which outlier handling method preserved data quality best?

**Answer:**

Winsorization preserved data quality best because it replaced extreme values with acceptable boundary values instead of removing rows. This kept the dataset size unchanged while reducing the effect of outliers.

---

#### ❓ 3. How did data cleaning improve dataset usability?
**Answer:**

Data cleaning improved the dataset by handling missing values and reducing the effect of outliers. This made the dataset more complete, consistent, reliable, and suitable for data analysis and machine learning models.